## BLC data Ingestion Framework
NoteBook: Silver_ingestion_rearc

| Author | Date | Change Record | Description |
| ----------- | ----------- | ----------- | -----------
|Hemant kaushik	|05-Nov-2026|	-	| Notebook for ingesting BLC data from BRNZ to SLVR
 



In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col,trim,current_timestamp,input_file_name,when,row_number,explode,trim,from_json
from pyspark.sql.window import Window
from pyspark.sql.types import StructType,StructField,StringType,DoubleType,LongType,ArrayType
from delta.tables import DeltaTable

In [0]:
base_path = "/Volumes/workspace/hemant_poc/volume_dev/sourcefiles/"
brnz_prefix = "workspace.bronze.bronze_pr_"
slvr_prefix = "workspace.silver.silver_pr_"

In [0]:
silver_config = [
    {
        "bronze_table": f"{brnz_prefix}series",
        "silver_table": f"{slvr_prefix}series",
        "key_columns": ["series_id"]
    },
    {
        "bronze_table": f"{brnz_prefix}sector",
        "silver_table": f"{slvr_prefix}sector",
        "key_columns": ["sector_code"]
    },
    {
        "bronze_table": f"{brnz_prefix}seasonal",
        "silver_table": f"{slvr_prefix}seasonal",
        "key_columns": ["seasonal_code"]
    },
    {
        "bronze_table": f"{brnz_prefix}period",
        "silver_table": f"{slvr_prefix}period",
        "key_columns": ["period"]
    },         
    {
        "bronze_table": f"{brnz_prefix}measure",
        "silver_table": f"{slvr_prefix}measure",
        "key_columns": ["measure_code"]
    },
    {
        "bronze_table": f"{brnz_prefix}footnote",
        "silver_table": f"{slvr_prefix}footnote",
        "key_columns": ["footnote_code"]
    },    
    {
        "bronze_table": f"{brnz_prefix}duration",
        "silver_table": f"{slvr_prefix}duration",
        "key_columns": ["duration_code"]
    },    
    {
        "bronze_table": f"{brnz_prefix}class",
        "silver_table": f"{slvr_prefix}class",
        "key_columns": ["class_code"]
    },    
    {
        "bronze_table": f"{brnz_prefix}data",
        "silver_table": f"{slvr_prefix}data",
        "key_columns": ["series_id","year","period"]
    }
]

In [0]:
def load_silver(
    bronze_table,
    silver_table,
    keys
):
    # -----------------------------------
    # Read Bronze
    # -----------------------------------
    source_df = spark.table(bronze_table)
    # -----------------------------------
    # Basic DQ
    # -----------------------------------
    for key in keys:
        source_df = source_df.filter(
            col(key).isNotNull()
        )
    # -----------------------------------
    # First run: create table
    # -----------------------------------
    if not spark.catalog.tableExists(silver_table):
        (
            source_df
            .write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table)
        )

        print(f"Created: {silver_table}")

        return
    
    # -----------------------------------
    # Existing Silver
    # -----------------------------------

    target = DeltaTable.forName(
        spark,
        silver_table
    )


    # -----------------------------------
    # Dynamic MERGE key
    # -----------------------------------

    merge_condition = " AND ".join(
        [
            f"t.`{key}` <=> s.`{key}`"
            for key in keys
        ]
    )


    # -----------------------------------
    # Business columns
    # -----------------------------------

    ignore_columns = (
        keys
        + [
            "_ingested_at",
            "_source_file"
        ]
    )

    compare_columns = [
        c for c in source_df.columns if c not in ignore_columns
    ]


    # -----------------------------------
    # Detect actual changes
    # -----------------------------------

    change_condition = " OR ".join(
        [
            f"NOT (t.`{c}` <=> s.`{c}`)"
            for c in compare_columns
        ]
    )


    # -----------------------------------
    # MERGE
    # -----------------------------------

    (
        target.alias("t")
        .merge(source_df.alias("s"),merge_condition)
        .whenMatchedUpdateAll(condition=change_condition)
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Merged: {silver_table}")

In [0]:
for config in silver_config:

    create_silver_table(
        bronze_table=config["bronze_table"],
        silver_table=config["silver_table"],
        key_columns=config["key_columns"]
    )

In [0]:
population_schema = StructType([
    StructField(
        "data",
        ArrayType(
            StructType([
                StructField("Nation", StringType(), True),
                StructField("Nation ID", StringType(), True),
                StructField("Population", DoubleType(), True),
                StructField("Year", LongType(), True)
            ])
        ),
        True
    )
])

In [0]:
@dp.materialized_view(
    name="workspace.silver.silver_population",
    comment="Cleaned population data from Data USA"
)
def silver_population():

    df = spark.read.table("workspace.bronze.bronze_pr_population")

    # Step 1: Convert raw JSON string into structured JSON
    df = (
        df.withColumn("parsed_json",from_json(col("raw_json"),population_schema)
        )
    )

    # Step 2: Explode only the data array

    df = (
        df
        .select(
            explode(col("parsed_json.data")).alias("record"),
            col("_source_file"),col("_ingested_at")
        )
    )
    df = df.filter(col('record').isNotNull())

    df = (
        df
        .select(
            col("record.Year").cast("int").alias("year"),
            trim(col("record.Nation")).alias("nation"),
            trim(col("record.`Nation ID`")).alias("nation_id"),
            col("record.Population").cast("long").alias("population"),
            col("_source_file"),
            col("_ingested_at")
        )
    )

    # Basic DQ without @dp.expect_or_drop
    df = (
        df
        .filter(col("year").isNotNull())
        .filter(col("nation_id").isNotNull())
        .filter(col("population").isNotNull())
    )

    return df